In [ ]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [ ]:
# MAGIC %run /Workspace/Users/pawanvirat32@gmail.com/consolidated_pipeline/1_setup/utilities

In [ ]:
df = spark.read.options(header = True, inferSchema = True).csv(f'{processed_path}/*.csv').withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
df.show(5)

In [ ]:
dbutils.widgets.text("catalog", "fmcg", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://sportsbar-pr1/{data_source}'
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"
print("Base Path: ", base_path)
print("Landing Path: ", landing_path)
print("Processed Path: ", processed_path)


# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [ ]:
df = spark.read.options(header = True, inferSchema = True).csv(f'{landing_path}/*.csv').withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

print("Total Rows: ", df.count())
df.show(5)

In [ ]:
display(df.limit(30))

In [ ]:
df.write\
 .format("delta") \
 .option("delta.enableChangeDataFeed", "true") \
 .mode("append") \
 .saveAsTable(bronze_table)

In [ ]:
files = dbutils.fs.ls(landing_path)
display(files)

In [ ]:
for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

In [ ]:
df_orders = spark.sql(f'select * from {bronze_table}')
display(df_orders.limit(40))

In [ ]:
# keep rows with valid order id
df_orders = df_orders.filter(F.col("order_id").isNotNull())

# keep only numeric customer id else set it to 999999
df_orders = df_orders.withColumn(
    'customer_id',
    F.when(F.col('customer_id').rlike('^[0-9]+$'), F.col('customer_id'))
    .otherwise('999999')
    .cast('string')
)


# tuesday, date in date
df_orders = df_orders.withColumn(
    'order_placement_date',
    F.regexp_replace(F.col('order_placement_date'), r'^[A-Za-z]+,\s*', '')
)

# parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# drop duplicates
df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))

In [ ]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [ ]:
display(df_orders.limit(40))

In [ ]:
df_products = spark.table('fmcg.silver.products')

display(df_products.limit(49))

In [ ]:
df_joined = df_orders.join(df_products, on= 'product_id', how= 'inner').select(df_orders["*"], df_products["product_code"])

df_joined.show(5)

In [ ]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format('delta').option(
        'delta.enableChangeDataFeed', 'true'
    ).option('mergeSchema', 'true').mode('overwrite').saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias('silver').merge(df_joined.alias('bronze'), 'silver.order_placement = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()')

## Gold

In [ ]:
df_gold = spark.sql(f'select order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity from {silver_table};')
display(df_gold.limit(40))

In [ ]:
if not (spark.catalog.tableExists(gold_table)):
    print("creating New Table")
    df_gold.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
else:
    gold_delta = DeltaTable.forName(spark, gold_table)
    gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

## merge with parent company

- Note: We want data for monthly level but child data is on daily level

In [ ]:
df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
df_child.show(10)

In [ ]:
df_monthly = (
    df_child
    # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
    .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

    # 2.Group at monthly grain by month_start + product_code + customer_code
    .groupBy("month_start", "product_code", "customer_code")
    .agg(
        F.sum("sold_quantity").alias("sold_quantity")
    )

    # 3. Rename month_start back to `date` to match your target schema
    .withColumnRenamed("month_start", "date")
)

df_monthly.show(5, truncate=False)

In [ ]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()